In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib
import time

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)

# ------------------------------
# 1. Load and preprocess the MNIST dataset
# ------------------------------
def load_and_preprocess_data():
    # Load training and test data from Kaggle MNIST Digit Recognizer dataset
    train_data = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
    test_data = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')
    
    print("Training Data Shape:", train_data.shape)
    print("Test Data Shape:", test_data.shape)
    print("\nTraining Data (First 5 rows):\n", train_data.head())
    
    # Separate labels and features
    y_train = train_data['label'].values
    x_train = train_data.drop(columns=['label']).values
    x_test = test_data.values
    
    # Normalize pixel values from [0,255] to [0,1]
    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0
    
    # Split training data for validation
    x_train, x_val, y_train, y_val = train_test_split(
        x_train, y_train, test_size=0.2, random_state=42
    )
    
    return (x_train, y_train), (x_val, y_val), x_test


In [ ]:
# ------------------------------
# 2. Dimensionality Reduction with PCA
# ------------------------------
def apply_pca(x_train, x_val, x_test, n_components=0.95):
    # Apply PCA to reduce dimensionality while preserving 95% of variance
    pca = PCA(n_components=n_components, random_state=42)
    
    x_train_pca = pca.fit_transform(x_train)
    x_val_pca = pca.transform(x_val)
    x_test_pca = pca.transform(x_test) if x_test is not None else None
    
    print(f"Reduced dimensions from {x_train.shape[1]} to {x_train_pca.shape[1]}")
    print(f"Explained variance ratio: {sum(pca.explained_variance_ratio_):.4f}")
    
    return x_train_pca, x_val_pca, x_test_pca, pca

In [ ]:
# ------------------------------
# 3. Train and Evaluate Multiple Models
# ------------------------------
def train_and_evaluate_models(x_train, y_train, x_val, y_val):
    models = {
        'SVM': SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
        'KNN': KNeighborsClassifier(n_neighbors=3),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\nTraining {name}...")
        start_time = time.time()
        
        # Train model
        model.fit(x_train, y_train)
        
        # Make predictions
        y_pred = model.predict(x_val)
        
        # Calculate accuracy
        accuracy = accuracy_score(y_val, y_pred)
        train_time = time.time() - start_time
        
        # Store results
        results[name] = {
            'model': model,
            'accuracy': accuracy,
            'train_time': train_time,
            'predictions': y_pred
        }
        
        print(f"{name} Accuracy: {accuracy:.4f}")
        print(f"{name} Training Time: {train_time:.2f} seconds")
    
    return results


In [ ]:
# ------------------------------
# 4. Hyperparameter Tuning for Best Model
# ------------------------------
def tune_svm_hyperparameters(x_train, y_train):
    print("\nTuning SVM hyperparameters...")
    
    # Define parameter grid
    param_grid = {
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
        'kernel': ['rbf', 'poly']
    }
    
    # Create SVM classifier
    svm = SVC(random_state=42)
    
    # Perform grid search with cross-validation
    grid_search = GridSearchCV(
        estimator=svm, 
        param_grid=param_grid, 
        cv=3, 
        n_jobs=-1, 
        verbose=1
    )
    
    # Fit grid search
    grid_search.fit(x_train, y_train)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
    
    return grid_search.best_estimator_


In [ ]:
# ------------------------------
# 5. Save Model and Results
# ------------------------------
def save_model_and_results(model, model_name, accuracy, pca=None):
    # Create filename with accuracy
    filename = f"{model_name.replace(' ', '_').lower()}_acc_{accuracy:.4f}.pkl"
    
    # Save model
    joblib.dump(model, filename)
    print(f"\nModel saved as {filename}")
    
    # Save PCA transformer if provided
    if pca:
        pca_filename = f"pca_transformer.pkl"
        joblib.dump(pca, pca_filename)
        print(f"PCA transformer saved as {pca_filename}")
    
    return filename


In [ ]:
# ------------------------------
# 6. Visualize Results
# ------------------------------
def visualize_results(results, y_val):
    # Extract accuracies and training times
    model_names = list(results.keys())
    accuracies = [results[name]['accuracy'] for name in model_names]
    training_times = [results[name]['train_time'] for name in model_names]
    
    # Create subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot accuracies
    bars = ax1.bar(model_names, accuracies)
    ax1.set_title('Model Accuracies')
    ax1.set_ylabel('Accuracy')
    ax1.set_ylim(0.9, 1.0)
    
    # Add value labels on bars
    for bar, accuracy in zip(bars, accuracies):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.001,
                f'{accuracy:.4f}', ha='center', va='bottom')
    
    # Plot training times
    bars = ax2.bar(model_names, training_times)
    ax2.set_title('Training Times')
    ax2.set_ylabel('Time (seconds)')
    
    # Add value labels on bars
    for bar, time_val in zip(bars, training_times):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.001,
                f'{time_val:.2f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig('model_comparison.png')
    plt.show()
    
    # Plot confusion matrix for the best model
    best_model_name = max(results, key=lambda x: results[x]['accuracy'])
    y_pred = results[best_model_name]['predictions']
    
    cm = confusion_matrix(y_val, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix for {best_model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig('confusion_matrix.png')
    plt.show()


In [ ]:
# ------------------------------
# Main execution
# ------------------------------
def main():
    # Load and preprocess data
    (x_train, y_train), (x_val, y_val), x_test = load_and_preprocess_data()
    
    # Apply PCA for dimensionality reduction
    x_train_pca, x_val_pca, x_test_pca, pca = apply_pca(x_train, x_val, x_test)
    
    # Train and evaluate multiple models
    results = train_and_evaluate_models(x_train_pca, y_train, x_val_pca, y_val)
    
    # Identify best model
    best_model_name = max(results, key=lambda x: results[x]['accuracy'])
    best_accuracy = results[best_model_name]['accuracy']
    best_model = results[best_model_name]['model']
    
    print(f"\nBest model: {best_model_name} with accuracy: {best_accuracy:.4f}")
    
    # If SVM is the best model, tune its hyperparameters
    if best_model_name == 'SVM':
        print("\nSince SVM is the best model, let's tune its hyperparameters...")
        tuned_svm = tune_svm_hyperparameters(x_train_pca, y_train)
        
        # Evaluate tuned SVM
        tuned_accuracy = accuracy_score(y_val, tuned_svm.predict(x_val_pca))
        print(f"Tuned SVM Accuracy: {tuned_accuracy:.4f}")
        
        # Use tuned model if it's better
        if tuned_accuracy > best_accuracy:
            best_model = tuned_svm
            best_accuracy = tuned_accuracy
            print("Using tuned SVM as the best model.")
    
    # Save the best model
    model_filename = save_model_and_results(best_model, best_model_name, best_accuracy, pca)
    
    # Visualize results
    visualize_results(results, y_val)
    
    # Generate classification report for the best model
    y_pred = best_model.predict(x_val_pca)
    print(f"\nClassification Report for {best_model_name}:")
    print(classification_report(y_val, y_pred))
    
    # If test data is available, generate predictions
    if x_test is not None:
        print("\nGenerating predictions for test data...")
        x_test_pca = pca.transform(x_test)
        test_predictions = best_model.predict(x_test_pca)
        
        # Create submission file
        submission = pd.DataFrame({
            'ImageId': range(1, len(test_predictions) + 1),
            'Label': test_predictions
        })
        submission.to_csv('digit_recognizer_submission.csv', index=False)
        print("Submission file saved as 'digit_recognizer_submission.csv'")

if __name__ == '__main__':
    main()